# Marketing LLM — Adaptive LoRA Fine-Tune on Kaggle

**Auto-detects GPU and picks the optimal path:**
- **T4 ×2 / V100 / A100** (compute capability ≥ 7.0) → Unsloth + Llama 3.1 **8B** + 4-bit QLoRA
- **P100** (compute capability 6.0) → vanilla transformers + Llama 3.2 **3B** + fp16 LoRA

Either path saves a LoRA adapter and pushes to Hugging Face Hub. Output repo name reflects the actual model trained.

Set HF_TOKEN as a Kaggle Secret (Add-ons → Secrets) to enable HF push.

In [ ]:
# ── GPU detection + dependency install ────────────────────────
# CRITICAL: detect GPU WITHOUT importing torch, so we can install the right
# torch version BEFORE first import. (Pascal P100 needs torch 2.4.1; Kaggle's
# default torch 2.10 dropped cc 6.0 kernels — this is the bug that killed
# 8 previous runs.)
import subprocess, sys, os, re

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()
def pip(*a):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*a])
def pip_uninstall(*a):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y',*a], check=False)

GPU_NAME = sh('nvidia-smi --query-gpu=name --format=csv,noheader').splitlines()[0].strip()
IS_PASCAL = bool(re.search(r'P100|P40|P4(?!d)', GPU_NAME))
USE_UNSLOTH = not IS_PASCAL
PATH = 'unsloth-8b' if USE_UNSLOTH else 'vanilla-3b'

# Detect platform
if os.path.exists('/kaggle/working'):
    WORK_DIR, PLATFORM = '/kaggle/working', 'Kaggle'
elif os.path.exists('/content'):
    WORK_DIR, PLATFORM = '/content', 'Colab'
else:
    WORK_DIR, PLATFORM = os.getcwd(), 'Local'

print(f'Platform: {PLATFORM}  |  GPU: {GPU_NAME}  |  PATH: {PATH}')
print(f'(Detected via nvidia-smi, torch not yet loaded)')

pip('--upgrade','pip')

if USE_UNSLOTH:
    # T4×2 / V100 / A100 path — keep Kaggle's torch, add Unsloth + 4-bit
    pip('--upgrade','torchao>=0.16.0')
    pip_uninstall('unsloth','unsloth_zoo','bitsandbytes')
    pip('--upgrade','--no-cache-dir','bitsandbytes>=0.46.1')
    pip('--upgrade','transformers>=4.49.0,<4.55.0')
    pip('--upgrade','peft>=0.14.0,<0.16.0','trl>=0.12.0,<0.13.0','accelerate>=1.2.0')
    pip('--upgrade','--no-cache-dir',
        'git+https://github.com/unslothai/unsloth.git@main',
        'git+https://github.com/unslothai/unsloth_zoo.git@main')
else:
    # P100 path — PIN TORCH 2.4.1 (last release with Pascal cc 6.0 kernels).
    # Must happen BEFORE first 'import torch' anywhere in the notebook.
    print('Pinning torch 2.4.1 for Pascal (cc 6.0) compatibility…')
    pip_uninstall('torchao','torch','torchvision','torchaudio')
    pip('--upgrade','--no-cache-dir',
        'torch==2.4.1','torchvision==0.19.1','torchaudio==2.4.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
    pip('--upgrade','transformers>=4.46.0,<4.50.0')
    pip('--upgrade','peft>=0.11.0,<0.14.0')
    pip('--upgrade','trl>=0.10.0,<0.13.0','accelerate>=1.0.0')
    # Force-reset the import cache so any later 'import torch' gets the new wheel
    import importlib, sys as _sys
    for mod in list(_sys.modules):
        if mod == 'torch' or mod.startswith('torch.'):
            del _sys.modules[mod]

pip('datasets','huggingface_hub')

# Now safe to import torch — pip just (re)installed the right version
import torch
assert torch.cuda.is_available(), '❌ No GPU detected after install'
props = torch.cuda.get_device_properties(0)
CC_MAJOR, CC_MINOR = props.major, props.minor
VRAM_GB = props.total_memory / 1e9
print(f'✓ torch {torch.__version__} CUDA {torch.version.cuda}')
print(f'✓ {torch.cuda.get_device_name(0)} cc {CC_MAJOR}.{CC_MINOR}  {VRAM_GB:.1f} GB VRAM')
print(f'✓ Working dir: {WORK_DIR}')


In [ ]:
# Install was done in cell 1 (before torch import). This cell now just confirms.
print(f"✓ Ready: PATH={PATH}, torch={torch.__version__}")

In [ ]:
# ── Pull training data (Supabase live corpus + static seed) ─────
import os, json, urllib.request, subprocess

SUPABASE_URL = os.environ.get('SUPABASE_URL')
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY')

examples = []
if SUPABASE_URL and SUPABASE_KEY:
    try:
        req = urllib.request.Request(
            f'{SUPABASE_URL}/rest/v1/training_pairs?select=intent,instruction,output&order=created_at.desc&limit=10000',
            headers={'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'},
        )
        with urllib.request.urlopen(req, timeout=30) as r:
            rows = json.loads(r.read().decode())
        examples = [{'intent': r['intent'], 'instruction': r['instruction'], 'output': r['output']} for r in rows]
        print(f'✓ Pulled {len(examples)} training pairs from Supabase live corpus')
    except Exception as e:
        print(f'Supabase pull failed ({e}) — falling back to static seed')

# Clone the repo for the static seed fallback
REPO_DIR = f'{WORK_DIR}/Marketing'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/amittomar-hue/Marketing.git', REPO_DIR], check=True)

SEED_PATH = f'{REPO_DIR}/training/data/marketing_sft.jsonl'
if os.path.exists(SEED_PATH):
    with open(SEED_PATH) as f:
        seed = [json.loads(l) for l in f]
    print(f'+ {len(seed)} static seed examples merged')
    examples.extend(seed)

# Dedupe by instruction
seen = set()
deduped = []
for ex in examples:
    key = ex['instruction'].strip().lower()[:200]
    if key in seen: continue
    seen.add(key)
    deduped.append(ex)
examples = deduped

assert len(examples) >= 50, f'Need at least 50 training examples, got {len(examples)}'
print(f'Final training corpus: {len(examples)} examples')

In [ ]:
# ── Load base model (path-dependent) ────────────────────────────
import torch

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
    MODEL_SHORT = '8b'
    max_seq_length = 2048
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # Try unsloth's mirror first (always public), fall back to Meta's gated repo
    candidates = ['unsloth/Llama-3.2-3B-Instruct', 'meta-llama/Llama-3.2-3B-Instruct']
    model, tokenizer = None, None
    for name in candidates:
        try:
            print(f'Trying {name}...')
            tokenizer = AutoTokenizer.from_pretrained(name)
            model = AutoModelForCausalLM.from_pretrained(
                name,
                torch_dtype=torch.float16,
                device_map='auto',
                low_cpu_mem_usage=True,
            )
            MODEL_NAME = name
            MODEL_SHORT = '3b'
            max_seq_length = 1024
            print(f'✓ Loaded {name}')
            break
        except Exception as e:
            print(f'  failed: {str(e)[:200]}')
    assert model is not None, 'Could not load 3B model from any candidate'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Attach LoRA adapters ────────────────────────────────────────
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
else:
    from peft import LoraConfig, get_peft_model, TaskType
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none',
    )
    model = get_peft_model(model, cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# ── Format dataset with Llama 3 chat template ───────────────────
from datasets import Dataset

SYSTEM_PROMPT = 'You are Marketing LLM, an enterprise-grade marketing assistant. Be direct, specific, and data-driven. Format responses in clean markdown.'

def format_example(ex):
    messages = [
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':ex['instruction']},
        {'role':'assistant','content':ex['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = Dataset.from_list(examples)
ds = raw.map(format_example, remove_columns=raw.column_names)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds["train"])}, Eval: {len(ds["test"])}')

In [ ]:
# ── Save adapter ────────────────────────────────────────────────
import gc, torch, os
gc.collect(); torch.cuda.empty_cache()

ADAPTER_DIR = f'{WORK_DIR}/marketing-llm-{MODEL_SHORT}-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
os.system(f'du -sh {ADAPTER_DIR}')
print(f'✓ Saved to {ADAPTER_DIR}')

In [ ]:
# ── Push to Hugging Face Hub ────────────────────────────────────
# Token sources, in order:
# 1. HF_TOKEN env var (Colab Secrets or local env)
# 2. Kaggle Secrets (Kaggle only)
# 3. None (prints local download path)

hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass

if not hf_token and PLATFORM == 'Colab':
    try:
        from google.colab import userdata  # type: ignore
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if hf_token:
    from huggingface_hub import login, HfApi
    login(token=hf_token)
    api = HfApi(token=hf_token)
    REPO_NAME = f'dmoop/marketing-llm-{MODEL_SHORT}-lora'
    try:
        api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)
        model.push_to_hub(REPO_NAME, token=hf_token)
        tokenizer.push_to_hub(REPO_NAME, token=hf_token)
        print(f'\n✓ Pushed to https://huggingface.co/{REPO_NAME}')
    except Exception as e:
        print(f'HF push failed: {e}')
        print(f'Adapter saved at {ADAPTER_DIR} for manual download')
else:
    print(f'No HF_TOKEN found.')
    print(f'Adapter saved at {ADAPTER_DIR} — download via UI')

In [ ]:
# ── Push to Hugging Face Hub ────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    hf_token = None
    print(f'HF_TOKEN secret not available: {e}')

if hf_token:
    from huggingface_hub import login, HfApi
    login(token=hf_token)
    api = HfApi(token=hf_token)
    REPO_NAME = f'dmoop/marketing-llm-{MODEL_SHORT}-lora'
    try:
        api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)
        model.push_to_hub(REPO_NAME, token=hf_token)
        tokenizer.push_to_hub(REPO_NAME, token=hf_token)
        print(f'\n✓ Pushed to https://huggingface.co/{REPO_NAME}')
    except Exception as e:
        print(f'HF push failed: {e}')
        print(f'Adapter saved at {ADAPTER_DIR} for manual download via Kaggle output API')
else:
    print(f'Adapter saved at {ADAPTER_DIR} — no HF push attempted')